# Session 7 — Ensemble Methods
## ECAM Brussels · Machine Learning 2025 · Master 1 Business Analysis

**Three parts:**
1. 🌿 **Bagging** — parallel trees on bootstrap samples (reduces variance)
2. 🌲 **Random Forest** — bagging + random feature subsets (forces diversity)
3. 🚀 **Boosting** — sequential error correction (reduces bias)

**Running case study:** BankEU — predicting loan defaults.

---
> **How to use this notebook:**
> - Follow the **guided sections** with the instructor
> - Complete the **🔧 semi-guided** blocks with hints
> - Attempt the **🏋️ independent exercises** on your own
> - Solutions are provided in collapsed cells (don't peek too soon!)


In [ ]:
# ── Setup: install and import everything needed ──────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("✅ XGBoost available")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost not installed — run: pip install xgboost")

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
    print("✅ LightGBM available")
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("⚠️  LightGBM not installed — run: pip install lightgbm")

# Consistent plot style
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2E75B6', '#1F6B3A', '#C45A00', '#5A3E8A', '#C0392B']
print("\n✅ All libraries loaded. Ready to build ensembles!")


---
## Part 1 — Bagging: Bootstrap Aggregating

### 1.1 — The Instability Problem (WHY)

Before we build anything, let's *see* the problem ensembles solve.

**BankEU dataset:** We have loan applications with Income, Debt, and Age.
A single decision tree is dangerously unstable — change just a few rows and
the entire structure changes.


In [ ]:
# ── Demonstrate single-tree instability ──────────────────────────────────────
np.random.seed(42)

# Synthetic BankEU-style dataset: 200 loan applications
n = 200
income  = np.random.choice(['High', 'Low'],  n, p=[0.5, 0.5])
debt    = np.random.choice(['Low',  'High'], n, p=[0.55, 0.45])
age     = np.random.choice(['Old',  'Young'],n, p=[0.5, 0.5])

# Label: approve if (High income AND Low debt) OR (Old AND Low debt)
y = np.array([
    1 if (inc == 'High' and dbt == 'Low') or (ag == 'Old' and dbt == 'Low')
    else 0
    for inc, dbt, ag in zip(income, debt, age)
]).astype(float)
# Add noise
noise_idx = np.random.choice(n, 20, replace=False)
y[noise_idx] = 1 - y[noise_idx]

# Encode to numeric
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X = pd.DataFrame({
    'Income': le.fit_transform(income),   # High=0, Low=1
    'Debt':   le.fit_transform(debt),     # High=0, Low=1
    'Age':    le.fit_transform(age),      # Old=0, Young=1
})
feature_names = ['Income', 'Debt', 'Age']
y = y.astype(int)

# Train TWO trees on slightly different subsets (drop last 10 rows vs first 10)
subsetA = list(range(0,  180))   # rows 0–179
subsetB = list(range(20, 200))   # rows 20–199

treeA = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X.iloc[subsetA], y[subsetA])
treeB = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X.iloc[subsetB], y[subsetB])

from sklearn.tree import export_text
print("=== TREE A (trained on rows 0–179) ===")
print(export_text(treeA, feature_names=feature_names))
print("=== TREE B (trained on rows 20–199) ===")
print(export_text(treeB, feature_names=feature_names))

rootA = feature_names[treeA.tree_.feature[0]]
rootB = feature_names[treeB.tree_.feature[0]]
print(f"\n🔍 Tree A root split: {rootA}")
print(f"🔍 Tree B root split: {rootB}")
if rootA != rootB:
    print("⚠️  Root split changed! This is HIGH VARIANCE.")
else:
    print("✅  Root split is the same (both trees happen to agree here).")


### 1.2 — Bootstrap Sampling: Step by Step (WHAT)

Bootstrap sampling is the trick that creates **diversity from one dataset**.

We draw N samples **with replacement** — so some rows appear twice, some not at all.


In [ ]:
# ── Visualise bootstrap sampling ─────────────────────────────────────────────
np.random.seed(0)

N = 10
original = np.arange(1, N+1)   # rows 1..10

fig, axes = plt.subplots(1, 4, figsize=(14, 3))

axes[0].bar(original, np.ones(N), color='#2E75B6', edgecolor='white', linewidth=1.5)
axes[0].set_title("Original dataset\n(rows 1–10)", fontsize=11, fontweight='bold')
axes[0].set_xticks(original)
axes[0].set_xlabel("Row index"); axes[0].set_ylabel("Count")

for i in range(1, 4):
    sample = np.random.choice(original, N, replace=True)
    counts = np.bincount(sample, minlength=N+1)[1:]
    colors = ['#C45A00' if c > 1 else ('#2E75B6' if c == 1 else '#E0E0E0')
              for c in counts]
    axes[i].bar(original, counts, color=colors, edgecolor='white', linewidth=1.5)
    axes[i].set_title(f"Bootstrap Sample {i}\n"
                      f"({sum(counts>0)} unique, {sum(counts==0)} OOB)",
                      fontsize=11, fontweight='bold')
    axes[i].set_xticks(original)
    axes[i].set_yticks([0, 1, 2, 3])
    axes[i].set_xlabel("Row index")
    axes[i].set_ylim(0, 3.5)

orange_patch = mpatches.Patch(color='#C45A00', label='Appears 2+ times')
blue_patch   = mpatches.Patch(color='#2E75B6', label='Appears once')
gray_patch   = mpatches.Patch(color='#E0E0E0', label='OOB (not drawn)')
fig.legend(handles=[orange_patch, blue_patch, gray_patch],
           loc='lower center', ncol=3, fontsize=10, frameon=True)
plt.suptitle("Bootstrap Sampling — Each sample draws N=10 rows WITH replacement",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('bootstrap_sampling.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key: orange = duplicated row, grey = OOB row")


In [ ]:
# ── Prove the 63.2% rule empirically ────────────────────────────────────────
np.random.seed(42)

dataset_sizes = [10, 50, 100, 500, 1000, 5000]
n_trials = 2000
results = []

for N in dataset_sizes:
    inclusion_rates = []
    for _ in range(n_trials):
        sample = np.random.choice(N, N, replace=True)
        unique_fraction = len(np.unique(sample)) / N
        inclusion_rates.append(unique_fraction)
    mean_rate = np.mean(inclusion_rates)
    results.append((N, mean_rate))
    print(f"N={N:5d}: mean unique fraction = {mean_rate:.4f}  "
          f"(theoretical 1-1/e = {1 - 1/np.e:.4f})")

print(f"\n📐 Mathematical limit: 1 - 1/e = {1 - 1/np.e:.6f}")
print("The convergence is fast — even at N=10 we're close to 63.2%!")


### 1.3 — Implementing BaggingClassifier (HOW)

Now let's build a Bagging ensemble and compare it to a single tree.
We use the **breast cancer** dataset — a real medical classification problem
(569 patients, 30 features, binary label: malignant/benign).


In [ ]:
# ── Load real dataset ────────────────────────────────────────────────────────
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
feature_names_bc = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=42
)

print(f"Dataset: {data.target_names[0]} vs {data.target_names[1]}")
print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Class balance (train): {np.bincount(y_train)}")


In [ ]:
# ── Single tree vs Bagging: side-by-side comparison ─────────────────────────
single_tree = DecisionTreeClassifier(max_depth=None, random_state=42)

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=None),
    n_estimators=100,    # 100 trees
    max_samples=1.0,     # each sample = N rows (bootstrap)
    max_features=1.0,    # ALL features (pure bagging, not RF yet)
    bootstrap=True,      # sampling WITH replacement
    oob_score=True,      # free validation on the ~37% OOB rows
    n_jobs=-1,
    random_state=42
)

# Cross-validated AUC (5-fold) — the honest comparison metric
cv_single = cross_val_score(single_tree, X_bc, y_bc, cv=5, scoring='roc_auc')
cv_bag    = cross_val_score(bag,         X_bc, y_bc, cv=5, scoring='roc_auc')

print("── Cross-validated AUC-ROC (5-fold) ──────────────────────────────────")
print(f"Single Tree : {cv_single.mean():.4f} ± {cv_single.std():.4f}")
print(f"Bagging     : {cv_bag.mean():.4f}    ± {cv_bag.std():.4f}")
print(f"\nGain from bagging: +{cv_bag.mean() - cv_single.mean():.4f}")

# Fit bagging to get OOB score
bag.fit(X_train, y_train)
print(f"\n── OOB Score (free validation, no extra fitting) ──────────────────────")
print(f"OOB score: {bag.oob_score_:.4f}")
print(f"Test score: {bag.score(X_test, y_test):.4f}")
print(f"Difference OOB vs test: {abs(bag.oob_score_ - bag.score(X_test, y_test)):.4f}")


In [ ]:
# ── How many trees do we actually need? Error vs n_estimators ────────────────
n_estimator_range = [1, 5, 10, 20, 50, 100, 200, 300, 500]
train_errors, test_errors = [], []

for n in n_estimator_range:
    b = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None),
        n_estimators=n, bootstrap=True, n_jobs=-1, random_state=42
    ).fit(X_train, y_train)
    train_errors.append(1 - b.score(X_train, y_train))
    test_errors.append(1 - b.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_estimator_range, train_errors, 'o-', color=COLORS[0],
        label='Train error', linewidth=2)
ax.plot(n_estimator_range, test_errors,  's-', color=COLORS[2],
        label='Test error',  linewidth=2)
ax.axhline(1 - cross_val_score(single_tree, X_bc, y_bc, cv=5).mean(),
           color='gray', linestyle='--', label='Single tree (CV mean)', linewidth=1.5)
ax.set_xlabel("Number of trees", fontsize=12)
ax.set_ylabel("Error rate", fontsize=12)
ax.set_title("Bagging: Error vs Number of Trees\n"
             "More trees → more stable. Error plateaus, never rises.", fontsize=12)
ax.legend(fontsize=10)
ax.set_xscale('log')
plt.tight_layout()
plt.savefig('bagging_n_estimators.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Observation: test error drops steeply then plateaus after ~50 trees.")
print("   Adding more trees NEVER increases test error in bagging.")


### 🔧 Semi-Guided Exercise 1.1 — Pasting vs Bagging

`BaggingClassifier` has a `bootstrap` parameter.
- `bootstrap=True` → **Bagging** (sample with replacement)
- `bootstrap=False` → **Pasting** (sample without replacement)

**Your task:**
1. Create a Pasting classifier (bootstrap=False, max_samples=0.7)
2. Compare its 5-fold CV AUC to Bagging
3. Which performs better? Why might that be?

💡 *Hint: without replacement, each tree sees a clean 70% of data — no duplicates.
With replacement, trees see ~63.2% unique rows but can have duplicates.*


In [ ]:
# ── Your code here ───────────────────────────────────────────────────────────
# 1. Create a Pasting classifier
pasting = BaggingClassifier(
    # TODO: fill in parameters
    # estimator = ?
    # n_estimators = ?
    # bootstrap = ?      ← this is the key parameter
    # max_samples = ?    ← use 0.7
    random_state=42, n_jobs=-1
)

# 2. Cross-validate both
# cv_pasting = cross_val_score(...)

# 3. Print and compare
# print(f"Bagging  AUC: {cv_bag.mean():.4f}")
# print(f"Pasting  AUC: {cv_pasting.mean():.4f}")


In [ ]:
# ── SOLUTION ─────────────────────────────────────────────────────────────────
pasting = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=None),
    n_estimators=100,
    bootstrap=False,      # ← Pasting: no replacement
    max_samples=0.7,      # each tree sees 70% of rows
    n_jobs=-1,
    random_state=42
)

cv_pasting = cross_val_score(pasting, X_bc, y_bc, cv=5, scoring='roc_auc')

print("── Bagging vs Pasting ─────────────────────────────────────────────────")
print(f"Bagging  (bootstrap=True,  max_samples=1.0): AUC = {cv_bag.mean():.4f} ± {cv_bag.std():.4f}")
print(f"Pasting  (bootstrap=False, max_samples=0.7): AUC = {cv_pasting.mean():.4f} ± {cv_pasting.std():.4f}")
print()
print("Why the difference?")
print("  Bagging: ~63.2% unique rows + duplicates → trees have more variation")
print("  Pasting: clean 70% subsets → less diversity between trees")
print("  In practice, results are dataset-dependent. Bagging is more common.")


---
## Part 2 — Random Forest: Bagging + Random Feature Subsets

### 2.1 — The Correlated-Trees Problem (WHY)

Plain Bagging still has a weakness: if one feature is very strong (e.g. Income),
**all 100 trees will use it as their first split**, making them highly correlated.

Random Forest's fix: at each node, only consider **√p randomly selected features**.
This forces trees to explore different patterns — they can't all default to Income.


In [ ]:
# ── Visualise the correlation problem ────────────────────────────────────────
# Show how feature usage changes with max_features

np.random.seed(42)
results_corr = {}

for label, mf in [('Bagging\n(max_features=1.0)', 1.0),
                   ('Random Forest\n(max_features=sqrt)', 'sqrt')]:
    root_features = []
    for seed in range(200):
        clf = DecisionTreeClassifier(max_features=mf, random_state=seed)
        bs_idx = np.random.choice(len(X_train), len(X_train), replace=True)
        clf.fit(X_train[bs_idx], y_train[bs_idx])
        root_features.append(clf.tree_.feature[0])  # which feature was used at root?
    results_corr[label] = root_features

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, features) in zip(axes, results_corr.items()):
    counts = np.bincount(features, minlength=X_train.shape[1])
    top10_idx = np.argsort(counts)[::-1][:10]
    ax.bar(range(10), counts[top10_idx], color=COLORS[0], edgecolor='white')
    ax.set_xticks(range(10))
    ax.set_xticklabels([feature_names_bc[i][:12] for i in top10_idx],
                       rotation=45, ha='right', fontsize=8)
    ax.set_ylabel("Times used as ROOT split (out of 200 trees)")
    ax.set_title(label, fontsize=11, fontweight='bold')

plt.suptitle("Root split feature distribution across 200 trees\n"
             "Left: Bagging concentrates on 1-2 features. Right: RF is much more diverse.",
             fontsize=11)
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.2 — Random Forest: Full Implementation (HOW)

Now let's build the complete Random Forest and extract everything it gives us:
**accuracy, OOB score, and feature importances**.


In [ ]:
# ── Full Random Forest ────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,        # 200 trees — more stable than default 100
    max_features='sqrt',     # √30 ≈ 5 features per split (classification default)
    max_depth=None,          # fully grown trees — intentional overfitting per tree
    min_samples_leaf=1,      # minimum 1 sample per leaf
    oob_score=True,          # enable free OOB validation
    n_jobs=-1,               # parallelise across all CPU cores
    random_state=42
)
rf.fit(X_train, y_train)

cv_rf = cross_val_score(rf, X_bc, y_bc, cv=5, scoring='roc_auc')

print("── Random Forest Results ──────────────────────────────────────────────")
print(f"CV AUC (5-fold) : {cv_rf.mean():.4f} ± {cv_rf.std():.4f}")
print(f"OOB Score       : {rf.oob_score_:.4f}")
print(f"Test Accuracy   : {rf.score(X_test, y_test):.4f}")
print(f"Test AUC        : {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}")
print()
print("── Comparison so far ──────────────────────────────────────────────────")
print(f"Single Tree : {cv_single.mean():.4f}")
print(f"Bagging     : {cv_bag.mean():.4f}")
print(f"Random Forest: {cv_rf.mean():.4f}  ← best so far")


### 2.3 — Feature Importance (MDI) (WHAT + Business Translation)

Random Forest computes **Mean Decrease in Impurity (MDI)**: for each feature,
sum the weighted Gini reduction across all splits on that feature, across all trees.

This tells you: *which features does the model rely on most?*


In [ ]:
# ── Feature importance: MDI vs Permutation Importance ─────────────────────────
importances_mdi  = pd.Series(rf.feature_importances_, index=feature_names_bc)
importances_mdi  = importances_mdi.sort_values(ascending=False)

# Permutation importance: shuffle feature F, measure AUC drop
perm_result = permutation_importance(rf, X_test, y_test,
                                     n_repeats=20, scoring='roc_auc',
                                     random_state=42, n_jobs=-1)
importances_perm = pd.Series(perm_result.importances_mean, index=feature_names_bc)
importances_perm = importances_perm.sort_values(ascending=False)

# Plot side by side: top 10
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top10_mdi  = importances_mdi.head(10)
top10_perm = importances_perm.head(10)

axes[0].barh(top10_mdi.index[::-1], top10_mdi.values[::-1],
             color=COLORS[0], edgecolor='white')
axes[0].set_xlabel("MDI Importance (normalized)", fontsize=11)
axes[0].set_title("MDI Feature Importance\n(Mean Decrease in Impurity)", fontsize=11, fontweight='bold')
axes[0].axvline(0, color='gray', linewidth=0.8)

axes[1].barh(top10_perm.index[::-1], top10_perm.values[::-1],
             color=COLORS[2], edgecolor='white',
             xerr=perm_result.importances_std[
                 [list(feature_names_bc).index(f) for f in top10_perm.index[::-1]]
             ], capsize=3)
axes[1].set_xlabel("AUC drop when feature is shuffled", fontsize=11)
axes[1].set_title("Permutation Importance\n(more reliable for high-cardinality features)", fontsize=11, fontweight='bold')

plt.suptitle("Two Ways to Measure Feature Importance\n"
             "MDI: how much each feature reduces impurity   |   "
             "Permutation: how much accuracy drops if feature is randomised",
             fontsize=11)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("── Top 5 features (MDI) ───────────────────────────────────────────────")
for feat, val in top10_mdi.head(5).items():
    print(f"  {feat:<35s}: {val:.4f}")
print()
print("⚠️  MDI can be biased toward high-cardinality features.")
print("   Permutation importance (right chart) is more reliable for reporting.")


In [ ]:
# ── BankEU-style business translation ────────────────────────────────────────
print("── 🏦 Business Translation (as if this were BankEU) ──────────────────")
print()
top3 = importances_perm.head(3)
for rank, (feat, val) in enumerate(top3.items(), 1):
    print(f"  #{rank} {feat}")
    print(f"     Randomising this feature drops AUC by {val:.4f}")
    print(f"     → This feature accounts for a significant portion of the model's predictive power.")
    print()
print("What to tell the credit committee:")
print(f"  'The model relies most heavily on: {top3.index[0]}.'")
print(f"  'This should be the focus of underwriting verification.'")
print(f"  'Session 8 (SHAP) will let us explain individual decisions per applicant.'")


### 🔧 Semi-Guided Exercise 2.1 — The max_features Tradeoff

`max_features` controls how many features each node can consider.
- **Lower value** → more diversity between trees, but more bias per tree
- **Higher value** → trees look more like bagging, less diverse

**Your task:**
1. Train RF models with max_features = [2, 'sqrt', 0.5, 0.8, None (=all)]
2. For each, record the 5-fold CV AUC
3. Plot the result
4. Which max_features is best here? Does it match the 'sqrt' default?

💡 *Hint: use a loop and `cross_val_score`*


In [ ]:
# ── Your code here ───────────────────────────────────────────────────────────
mf_options = [2, 'sqrt', 0.5, 0.8, None]
mf_scores  = []

for mf in mf_options:
    rf_test = RandomForestClassifier(
        n_estimators=100,
        # TODO: set max_features=mf
        oob_score=False,
        n_jobs=-1,
        random_state=42
    )
    # TODO: compute cv_score = cross_val_score(...)
    # mf_scores.append(cv_score.mean())
    pass

# TODO: plot mf_options vs mf_scores


In [ ]:
# ── SOLUTION ─────────────────────────────────────────────────────────────────
mf_options = [2, 'sqrt', 0.5, 0.8, None]
mf_labels  = ['2 features', 'sqrt (default)', '50% features', '80% features', 'All features']
mf_scores  = []

for mf in mf_options:
    rf_test = RandomForestClassifier(
        n_estimators=100, max_features=mf, n_jobs=-1, random_state=42
    )
    score = cross_val_score(rf_test, X_bc, y_bc, cv=5, scoring='roc_auc')
    mf_scores.append(score.mean())
    print(f"max_features={str(mf):<8}: AUC = {score.mean():.4f} ± {score.std():.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(mf_labels, mf_scores, color=[COLORS[1] if s == max(mf_scores) else COLORS[0]
                                            for s in mf_scores], edgecolor='white')
ax.set_ylim(min(mf_scores) - 0.01, max(mf_scores) + 0.01)
ax.set_ylabel("5-fold CV AUC")
ax.set_title("max_features Tradeoff: Diversity vs Per-Tree Accuracy", fontsize=12, fontweight='bold')
for bar, score in zip(bars, mf_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{score:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('max_features_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Part 3 — Boosting: Sequential Error Correction

### 3.1 — AdaBoost: Worked Example From Scratch (WHAT)

Let's implement AdaBoost **by hand** on the small 6-row BankEU example from the slides.
This makes the weight update mechanism fully transparent.


In [ ]:
# ── AdaBoost from scratch: 6 rows, 3 rounds ──────────────────────────────────
import pandas as pd

# Dataset: 6 loan applicants (from slides)
data_ada = pd.DataFrame({
    'Income': ['High','Low', 'High','High','Low', 'Low'],
    'Debt':   ['Low', 'High','High','Low', 'High','Low'],
    'Age':    ['Old', 'Old', 'Young','Old','Young','Young'],
    'Label':  [1,     0,     0,      0,    1,      1]
    # ↑ Rows 3 & 4 are deliberate noise: Row 3 = wealthy customer who previously defaulted;
    #   Row 4 = young applicant with guarantor. No single feature can separate perfectly.
})
# Encode
X_ada = pd.get_dummies(data_ada[['Income','Debt','Age']]).values.astype(float)
y_ada = np.array(data_ada['Label'].map({1:+1, 0:-1}))  # AdaBoost uses +1/-1

n = len(y_ada)
weights = np.full(n, 1/n)   # uniform starting weights: 1/6 each
alphas  = []
stumps  = []

print(f"{'Round':<8} {'Stump splits on':<20} {'Error ε':<12} {'Alpha α':<12} "
      f"{'Wrong rows':<15} {'Max weight after update':<25}")
print("-" * 100)

for round_num in range(1, 4):
    # Train a weighted stump (max_depth=1)
    stump = DecisionTreeClassifier(max_depth=1)
    stump.fit(X_ada, y_ada, sample_weight=weights)
    preds = stump.predict(X_ada)

    # Weighted error: sum of weights of misclassified examples
    incorrect = (preds != y_ada)
    eps = np.sum(weights[incorrect])

    # Model weight alpha
    eps = np.clip(eps, 1e-10, 1 - 1e-10)   # avoid log(0)
    alpha = 0.5 * np.log((1 - eps) / eps)

    # Update weights
    weights *= np.exp(-alpha * y_ada * preds)
    weights /= weights.sum()   # normalise

    alphas.append(alpha)
    stumps.append(stump)

    wrong_rows = np.where(incorrect)[0] + 1   # 1-indexed
    feat_map   = {0:'Income_Low', 1:'Income_High', 2:'Debt_Low', 3:'Debt_High', 4:'Age_Old', 5:'Age_Young'}
    feat_used  = feat_map.get(stump.tree_.feature[0], f'feat_{stump.tree_.feature[0]}')
    print(f"Round {round_num:<3} {feat_used:<20} {eps:<12.4f} {alpha:<12.4f} "
          f"{str(list(wrong_rows)):<15} {max(weights):<25.4f}")

print()
print("── Final prediction formula ────────────────────────────────────────────")
print("ŷ = sign( α₁·h₁(x) + α₂·h₂(x) + α₃·h₃(x) )")
print(f"   = sign( {alphas[0]:.3f}·h₁(x) + {alphas[1]:.3f}·h₂(x) + {alphas[2]:.3f}·h₃(x) )")
print()
print("A model with lower error gets higher α — it speaks LOUDER in the final vote.")


In [ ]:
# ── Visualise weight evolution ────────────────────────────────────────────────
# Run again collecting weights at each round
weights_history = [np.full(n, 1/n)]
w = np.full(n, 1/n)

for stump, alpha in zip(stumps, alphas):
    preds = stump.predict(X_ada)
    w = w * np.exp(-alpha * y_ada * preds)
    w /= w.sum()
    weights_history.append(w.copy())

rows_labels = ['Row ' + str(i+1) + ' (' + ('Approve' if l==1 else 'Reject') + ')' for i, l in enumerate(data_ada['Label'])]

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
round_titles = ['Round 1 (uniform)', 'After Round 1', 'After Round 2', 'After Round 3']

for ax, wts, title in zip(axes, weights_history, round_titles):
    colors = [COLORS[2] if wt == max(wts) else COLORS[0] for wt in wts]
    ax.bar(range(n), wts, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_xticks(range(n))
    ax.set_xticklabels(rows_labels, fontsize=8)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel("Weight" if ax == axes[0] else "")
    for j, wt in enumerate(wts):
        ax.text(j, wt + 0.003, f'{wt:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle("AdaBoost: Weight Evolution Across Rounds\n"
             "Orange = highest-weight row (hardest example — model must focus here)",
             fontsize=11)
plt.tight_layout()
plt.savefig('adaboost_weights.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.2 — AdaBoost in sklearn (HOW)


In [ ]:
# ── sklearn AdaBoost ─────────────────────────────────────────────────────────
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # stumps as base learners
    n_estimators=200,
    learning_rate=1.0,    # in AdaBoost, lr=1 is the standard default
    random_state=42
)

cv_ada = cross_val_score(ada, X_bc, y_bc, cv=5, scoring='roc_auc')
print(f"AdaBoost CV AUC: {cv_ada.mean():.4f} ± {cv_ada.std():.4f}")

# Accuracy vs number of estimators (staged_predict_proba)
ada.fit(X_train, y_train)
staged_auc = [
    roc_auc_score(y_test, proba[:, 1])
    for proba in ada.staged_predict_proba(X_test)
]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(staged_auc)+1), staged_auc, color=COLORS[2], linewidth=1.5)
ax.set_xlabel("Number of boosting rounds")
ax.set_ylabel("Test AUC")
ax.set_title("AdaBoost: AUC vs Number of Rounds\n"
             "Performance builds gradually — early rounds are weak", fontsize=11)
ax.axhline(rf.oob_score_, color=COLORS[0], linestyle='--', label=f'RF OOB ({rf.oob_score_:.3f})')
ax.legend()
plt.tight_layout()
plt.savefig('adaboost_staged.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.3 — Gradient Boosting: Fitting Residuals (WHAT)

AdaBoost reweights examples. **Gradient Boosting** takes a different approach:
each new tree explicitly fits the **residual errors** of the current ensemble.

Let's see this in action on a regression problem first — easiest to visualise.


In [ ]:
# ── Gradient boosting residuals: regression demo ─────────────────────────────
from sklearn.tree import DecisionTreeRegressor

np.random.seed(42)
X_reg = np.linspace(0, 10, 100).reshape(-1, 1)
y_reg = np.sin(X_reg.ravel()) + 0.3 * np.random.randn(100)

# Manual gradient boosting: 4 rounds
F = np.full(len(y_reg), y_reg.mean())   # F_0 = mean
eta = 0.8
trees_gb = []
residuals_history = [y_reg - F]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for m in range(4):
    residuals = y_reg - F
    tree = DecisionTreeRegressor(max_depth=3)
    tree.fit(X_reg, residuals)
    h = tree.predict(X_reg)
    F = F + eta * h
    trees_gb.append(tree)
    residuals_history.append(y_reg - F)

    # Top row: data + current prediction
    ax_top = axes[0, m]
    ax_top.scatter(X_reg, y_reg, s=10, color='gray', alpha=0.5, label='Data')
    ax_top.plot(X_reg, F, color=COLORS[2], linewidth=2,
                label=f'F_{m+1}(x)')
    ax_top.set_title(f"Round {m+1}: Prediction", fontsize=10, fontweight='bold')
    ax_top.legend(fontsize=8)

    # Bottom row: residuals this tree targets
    ax_bot = axes[1, m]
    ax_bot.scatter(X_reg, residuals, s=10, color=COLORS[0], alpha=0.5,
                   label='Residuals')
    ax_bot.plot(X_reg, h, color=COLORS[2], linewidth=2, label=f'Tree {m+1} fit')
    ax_bot.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax_bot.set_title(f"Round {m+1}: Residuals → Tree fit", fontsize=10, fontweight='bold')
    ax_bot.legend(fontsize=8)
    rmse = np.sqrt(np.mean((y_reg - F)**2))
    ax_bot.set_xlabel(f"RMSE after this round: {rmse:.4f}")

plt.suptitle("Gradient Boosting: Each tree fits the RESIDUALS of the current ensemble\n"
             "Top: prediction improves. Bottom: residuals shrink each round.",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('gradient_boosting_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key insight: each tree does NOT predict y — it predicts how much to CORRECT F.")
print(f"RMSE after 0 rounds (mean): {np.sqrt(np.mean((y_reg - y_reg.mean())**2)):.4f}")
print(f"RMSE after 4 rounds:        {np.sqrt(np.mean((y_reg - F)**2)):.4f}")


### 3.4 — Learning Rate and Early Stopping (WHAT + HOW)

The learning rate η controls how much of each tree's correction we apply.
Too high → overfits quickly. Too low → needs many trees.

**Early stopping** finds the optimal number of trees automatically.


In [ ]:
# ── Learning rate comparison ─────────────────────────────────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

learning_rates = [0.5, 0.1, 0.05, 0.01]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr, color in zip(learning_rates, COLORS):
    gb = GradientBoostingClassifier(
        n_estimators=300, learning_rate=lr,
        max_depth=3, random_state=42
    )
    gb.fit(X_tr, y_tr)

    train_scores = [roc_auc_score(y_tr,  p[:,1])
                    for p in gb.staged_predict_proba(X_tr)]
    val_scores   = [roc_auc_score(y_val, p[:,1])
                    for p in gb.staged_predict_proba(X_val)]
    best_round   = int(np.argmax(val_scores)) + 1

    axes[0].plot(train_scores, color=color, linewidth=1.5,
                 label=f'η={lr} (train)')
    axes[1].plot(val_scores,   color=color, linewidth=1.5,
                 label=f'η={lr} (best round={best_round})')

for ax, title in zip(axes, ['Train AUC', 'Validation AUC']):
    ax.set_xlabel("Number of trees"); ax.set_ylabel("AUC")
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle("Learning Rate Effect: lower η → needs more trees, but generalises better",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('learning_rate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("📌 Key observations:")
print("  • η=0.5: trains fast but validation peaks early (risks overfitting)")
print("  • η=0.1: good balance — recommended default")
print("  • η=0.01: very slow improvement — needs 1000+ trees")


In [ ]:
# ── XGBoost with early stopping ──────────────────────────────────────────────
if XGBOOST_AVAILABLE:
    from xgboost import XGBClassifier

    xgb = XGBClassifier(
        n_estimators=1000,       # set high — early stopping will find the right number
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,           # use 80% of rows per tree (like RF's bootstrap)
        colsample_bytree=0.8,    # use 80% of features per tree
        reg_alpha=0.0,           # L1 regularization (default 0)
        reg_lambda=1.0,          # L2 regularization (default 1)
        eval_metric='auc',
        early_stopping_rounds=20,
        random_state=42,
        verbosity=0
    )

    xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_val, y_val)],
        verbose=False
    )

    best_n = xgb.best_iteration + 1
    cv_xgb = cross_val_score(
        XGBClassifier(n_estimators=best_n, learning_rate=0.05,
                      max_depth=5, eval_metric='logloss',
                      random_state=42, verbosity=0),
        X_bc, y_bc, cv=5, scoring='roc_auc'
    )

    print(f"XGBoost: optimal n_estimators = {best_n} (found by early stopping)")
    print(f"XGBoost CV AUC: {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}")

    # Plot training curves
    results_dict = xgb.evals_result()
    rounds = range(1, len(results_dict['validation_0']['auc']) + 1)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(rounds, results_dict['validation_0']['auc'],
            color=COLORS[0], linewidth=1.5, label='Train AUC', alpha=0.7)
    ax.plot(rounds, results_dict['validation_1']['auc'],
            color=COLORS[2], linewidth=2,   label='Validation AUC')
    ax.axvline(best_n, color='black', linestyle='--', linewidth=1.5,
               label=f'Early stop at round {best_n}')
    ax.set_xlabel("Boosting round"); ax.set_ylabel("AUC")
    ax.set_title("XGBoost Early Stopping: Train vs Validation AUC\n"
                 "Model stops when validation stops improving for 20 rounds",
                 fontsize=11, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('xgb_early_stopping.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("XGBoost not available — install with: pip install xgboost")


### 3.5 — XGBoost vs LightGBM: Code Comparison (HOW)


In [ ]:
# ── Side-by-side: GBM vs XGBoost vs LightGBM ────────────────────────────────
import time

models_gb = {
    'sklearn GBM': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42
    ),
}

if XGBOOST_AVAILABLE:
    n_xgb = best_n if XGBOOST_AVAILABLE else 200
    models_gb['XGBoost'] = XGBClassifier(
        n_estimators=n_xgb, learning_rate=0.05, max_depth=5,
        eval_metric='logloss', random_state=42, verbosity=0
    )

if LIGHTGBM_AVAILABLE:
    from lightgbm import LGBMClassifier
    models_gb['LightGBM'] = LGBMClassifier(
        n_estimators=200, learning_rate=0.05, num_leaves=31,
        min_child_samples=20, random_state=42, verbose=-1
    )

print(f"{'Model':<20} {'CV AUC':<15} {'Std':<10} {'Train time (s)'}")
print("-" * 65)

for name, model in models_gb.items():
    t0 = time.time()
    cv = cross_val_score(model, X_bc, y_bc, cv=5, scoring='roc_auc')
    elapsed = time.time() - t0
    print(f"{name:<20} {cv.mean():<15.4f} ±{cv.std():<9.4f} {elapsed:.2f}s")


---
## Part 4 — Full Comparison: Which Method Wins?

Let's put all models head-to-head and build the full comparison table.


In [ ]:
# ── Grand comparison: all models ─────────────────────────────────────────────
all_models = {
    'Single Tree':  DecisionTreeClassifier(max_depth=None, random_state=42),
    'Bagging':      BaggingClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, oob_score=False,
                                            n_jobs=-1, random_state=42),
    'AdaBoost':     AdaBoostClassifier(n_estimators=200, random_state=42),
    'Gradient BM':  GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                               max_depth=3, random_state=42),
}

if XGBOOST_AVAILABLE:
    all_models['XGBoost'] = XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        eval_metric='logloss', random_state=42, verbosity=0
    )
if LIGHTGBM_AVAILABLE:
    all_models['LightGBM'] = LGBMClassifier(
        n_estimators=200, learning_rate=0.05, num_leaves=31,
        random_state=42, verbose=-1
    )

comparison = []
for name, model in all_models.items():
    cv = cross_val_score(model, X_bc, y_bc, cv=5, scoring='roc_auc')
    comparison.append({'Model': name, 'CV AUC Mean': cv.mean(), 'CV AUC Std': cv.std()})

df_comparison = pd.DataFrame(comparison).sort_values('CV AUC Mean', ascending=False)

print(df_comparison.to_string(index=False, float_format='{:.4f}'.format))

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = [COLORS[1] if m in ('XGBoost','LightGBM','Random Forest')
              else COLORS[0] for m in df_comparison['Model']]
bars = ax.barh(df_comparison['Model'], df_comparison['CV AUC Mean'],
               xerr=df_comparison['CV AUC Std'],
               color=colors_bar, edgecolor='white', capsize=4)
ax.set_xlabel("5-fold Cross-Validated AUC-ROC", fontsize=11)
ax.set_title("All Ensemble Methods: Head-to-Head Comparison\n"
             "Green = top performers", fontsize=12, fontweight='bold')
ax.set_xlim(0.92, 1.00)
for bar, val in zip(bars, df_comparison['CV AUC Mean']):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Variance comparison: distribution of 5 fold scores per model ─────────────
fig, ax = plt.subplots(figsize=(10, 5))
fold_scores_all = {}
for name, model in all_models.items():
    fold_scores_all[name] = cross_val_score(
        model, X_bc, y_bc, cv=10, scoring='roc_auc'
    )

ax.boxplot([fold_scores_all[n] for n in all_models],
           labels=list(all_models.keys()),
           patch_artist=True,
           boxprops=dict(facecolor=COLORS[0]+'55', color=COLORS[0]),
           medianprops=dict(color=COLORS[2], linewidth=2),
           whiskerprops=dict(color=COLORS[0]),
           capprops=dict(color=COLORS[0]))
ax.set_ylabel("AUC (each box = 10 folds)")
ax.set_title("Stability Comparison: 10-fold CV AUC Distribution\n"
             "Narrower box = more stable (lower variance)", fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('variance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Key insight: ensemble methods have narrower boxes than Single Tree.")
print("   This confirms: ensembles reduce variance.")


---
## 🏋️ Independent Exercises

### Exercise A — Random Forest Hyperparameter Tuning

Use the breast cancer dataset.

**Tasks:**
1. Try `min_samples_leaf` values: [1, 5, 10, 20, 50]
   - Plot CV AUC vs min_samples_leaf
   - What happens to training AUC vs test AUC as min_samples_leaf increases?

2. What does min_samples_leaf control? Explain in one sentence.

3. What is the optimal value for this dataset?


In [ ]:
# ── Exercise A — your code here ──────────────────────────────────────────────



In [ ]:
# ── Exercise A — Solution ────────────────────────────────────────────────────
min_leaf_vals = [1, 5, 10, 20, 50]
train_aucs, cv_aucs = [], []

for msl in min_leaf_vals:
    rf_a = RandomForestClassifier(
        n_estimators=100, min_samples_leaf=msl, n_jobs=-1, random_state=42
    )
    rf_a.fit(X_train, y_train)
    train_aucs.append(roc_auc_score(y_train, rf_a.predict_proba(X_train)[:,1]))
    cv_aucs.append(cross_val_score(rf_a, X_bc, y_bc, cv=5, scoring='roc_auc').mean())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(min_leaf_vals, train_aucs, 'o-', color=COLORS[0], linewidth=2, label='Train AUC')
ax.plot(min_leaf_vals, cv_aucs,   's-', color=COLORS[2], linewidth=2, label='CV AUC')
ax.set_xlabel("min_samples_leaf"); ax.set_ylabel("AUC")
ax.set_title("min_samples_leaf: bias–variance tradeoff", fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('exercise_a.png', dpi=150, bbox_inches='tight')
plt.show()

best_msl = min_leaf_vals[int(np.argmax(cv_aucs))]
print(f"Optimal min_samples_leaf = {best_msl}")
print()
print("min_samples_leaf controls the MINIMUM number of training samples allowed")
print("in a leaf node. Higher value → shallower trees → less overfit → more bias.")
print("Train AUC drops as min_samples_leaf increases (fewer perfect fits).")
print("CV AUC may improve slightly then drop (bias–variance tradeoff).")


### Exercise B — AdaBoost vs Gradient Boosting on Noisy Data

**Motivation:** AdaBoost amplifies the weight of hard examples.
If some examples are hard because they're *mislabelled*, AdaBoost will overfit to the noise.
Gradient Boosting with regularization is more robust.

**Tasks:**
1. Create a synthetic dataset with 10% label noise (flip 10% of labels randomly)
2. Train AdaBoost and Gradient Boosting on the noisy dataset
3. Evaluate both on a **clean** test set (no noise)
4. Which performs better on the clean test? Why?


In [ ]:
# ── Exercise B — your code here ──────────────────────────────────────────────



In [ ]:
# ── Exercise B — Solution ────────────────────────────────────────────────────
np.random.seed(42)

# Generate clean dataset
X_noisy, y_clean = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_redundant=2, random_state=42
)

# Add 10% label noise
y_noisy = y_clean.copy()
noise_idx = np.random.choice(len(y_noisy), int(0.10 * len(y_noisy)), replace=False)
y_noisy[noise_idx] = 1 - y_noisy[noise_idx]

# Train/test split (test set is clean — no noise added)
Xn_tr, Xn_te, yn_tr_noisy, yn_te_clean = train_test_split(
    X_noisy, y_clean, test_size=0.2, random_state=42
)
# Training labels are noisy
_, _, yn_tr_noisy, _ = train_test_split(
    X_noisy, y_noisy, test_size=0.2, random_state=42
)

ada_n = AdaBoostClassifier(n_estimators=200, random_state=42)
gbm_n = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                    max_depth=3, random_state=42)
rf_n  = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

results_noise = {}
for name, model in [('AdaBoost', ada_n), ('Gradient BM', gbm_n), ('Random Forest', rf_n)]:
    model.fit(Xn_tr, yn_tr_noisy)
    auc = roc_auc_score(yn_te_clean, model.predict_proba(Xn_te)[:,1])
    results_noise[name] = auc
    print(f"{name:<15}: Test AUC on clean data = {auc:.4f}")

print()
print("Interpretation:")
print("  AdaBoost amplifies noisy labels by increasing their weights → overfits noise")
print("  Gradient BM with small depth is more robust to noisy individual examples")
print("  Random Forest is most robust (averaging neutralises individual noisy trees)")


### Exercise C — BankEU Full Pipeline

Build a complete end-to-end pipeline for the BankEU loan classification problem.

**Tasks:**
1. Create a synthetic BankEU dataset with 5 features:
   Income (continuous), Debt Ratio (continuous), Age, Employment (years), Credit Score
2. Train Random Forest and XGBoost (if available)
3. Report: CV AUC, OOB score (RF), top-3 feature importances
4. Write 3 sentences explaining the results to the BankEU credit committee (non-technical language)


In [ ]:
# ── Exercise C — your code here ──────────────────────────────────────────────



In [ ]:
# ── Exercise C — Solution ────────────────────────────────────────────────────
np.random.seed(42)
n_bank = 1000

# Simulate BankEU loan data
income       = np.random.lognormal(mean=10.5, sigma=0.5, size=n_bank)   # €
debt_ratio   = np.random.beta(2, 5, size=n_bank)                         # 0–1
age          = np.random.randint(20, 70, size=n_bank)
employment_y = np.random.exponential(5, size=n_bank).clip(0, 40)
credit_score = np.random.normal(650, 80, size=n_bank).clip(300, 850)

# Normalise continuous features so coefficients work on similar scales
inc_norm  = (income - income.mean()) / income.std()
cs_norm   = (credit_score - credit_score.mean()) / credit_score.std()
age_norm  = (age - age.mean()) / age.std()
emp_norm  = (employment_y - employment_y.mean()) / employment_y.std()

# Default: approve if income high, debt low, credit score good
logit = (
    + 0.8  * inc_norm
    - 1.5  * debt_ratio
    + 0.6  * cs_norm
    - 0.3  * age_norm
    + 0.4  * emp_norm
    + 0.5  * np.random.randn(n_bank)   # noise
)
prob_approve = 1 / (1 + np.exp(-logit))
y_bank = (prob_approve > 0.5).astype(int)

X_bank = pd.DataFrame({
    'Income':       income,
    'Debt_Ratio':   debt_ratio,
    'Age':          age,
    'Employment_Yrs': employment_y,
    'Credit_Score': credit_score,
})
feat_bank = X_bank.columns.tolist()

Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    X_bank.values, y_bank, test_size=0.2, stratify=y_bank, random_state=42
)

# Train models
rf_bank = RandomForestClassifier(n_estimators=200, oob_score=True, n_jobs=-1, random_state=42)
rf_bank.fit(Xb_tr, yb_tr)
cv_bank = cross_val_score(rf_bank, X_bank.values, y_bank, cv=5, scoring='roc_auc')

print("── BankEU Model Results ────────────────────────────────────────────────")
print(f"CV AUC (5-fold):  {cv_bank.mean():.4f} ± {cv_bank.std():.4f}")
print(f"OOB Score:        {rf_bank.oob_score_:.4f}")
print(f"Test AUC:         {roc_auc_score(yb_te, rf_bank.predict_proba(Xb_te)[:,1]):.4f}")
print()

imp = pd.Series(rf_bank.feature_importances_, index=feat_bank).sort_values(ascending=False)
print("── Top Feature Importances ─────────────────────────────────────────────")
for feat, val in imp.items():
    print(f"  {feat:<20}: {val:.4f} ({val*100:.1f}%)")

print()
top3 = imp.head(3).index.tolist()
print("── 🏦 Credit Committee Briefing ────────────────────────────────────────")
print(f"  1. Our model achieves {cv_bank.mean():.1%} AUC on held-out data,")
print(f"     correctly ranking {cv_bank.mean():.1%} of applicant pairs by default risk.")
print(f"  2. The three most important factors are: {top3[0]}, {top3[1]}, and {top3[2]}.")
print(f"     {top3[0]} alone explains {imp[top3[0]]*100:.0f}% of the model's predictive power.")
print(f"  3. To explain WHY a specific applicant was rejected, we will use SHAP values")
print(f"     (Session 8) — which will provide individual, auditable justifications.")

# Feature importance chart
fig, ax = plt.subplots(figsize=(8, 4))
imp.plot.barh(ax=ax, color=[COLORS[1] if f in top3 else COLORS[0] for f in imp.index],
              edgecolor='white')
ax.set_xlabel("MDI Feature Importance"); ax.set_title("BankEU: Feature Importances", fontsize=11)
plt.tight_layout()
plt.savefig('bankeu_importances.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Session Summary

| Method | Mechanism | Reduces | Training | When to use |
|--------|-----------|---------|----------|-------------|
| **Bagging** | Bootstrap samples + vote | Variance | Parallel | Any high-variance base model |
| **Random Forest** | Bagging + random features | Variance | Parallel | Default baseline for tabular ML |
| **AdaBoost** | Reweight hard examples | Bias | Sequential | Clean data, need interpretable model |
| **Gradient BM** | Fit residuals (= gradient) | Bias | Sequential | When RF hits its ceiling |
| **XGBoost** | GB + regularisation + 2nd-order | Bias + overfit | Sequential | Production tabular ML |
| **LightGBM** | XGB + leaf-wise + histograms | Bias + speed | Sequential | Large datasets |

### The decision rule:
1. **Start with Random Forest** — fast, stable, gives OOB score and feature importance
2. **Switch to XGBoost** if you need more accuracy and can tune carefully
3. **Switch to LightGBM** if your dataset is very large (> 1M rows)
4. **Use single tree** only when you need full individual explanation (EU AI Act audit)

---
### Preview: Session 8 — Model Interpretability
Now that you can build powerful ensembles, the next challenge is explaining them.
**SHAP values** will let you answer: *why was applicant 42 specifically rejected?*
